# TECH 405 — Week 5 RNN Hackathon: SMS Spam Classifier (LSTM)

**Goal:** classify SMS messages as spam / ham with an LSTM (`nn.LSTM → nn.Linear`), trained from scratch (no pretrained models).

| Dataset | Classes | Baseline (majority) | Target |
|---|---|---|---|
| SMS Spam | spam / ham (2) | 86.6% | 96% |

## 1. Imports and setup

In [1]:
import re
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


## 2. Load the dataset

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv",
                 encoding="latin-1")[["v1","v2"]]        # v1 = ham/spam, v2 = text
df.columns = ["label", "text"]
df["y"] = (df["label"] == "spam").astype(int)

print(df.shape)
print(df["label"].value_counts())
df.head()

(5572, 3)
label
ham     4825
spam     747
Name: count, dtype: int64


,label,text,y
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## 3. Train / validation / test split (stratified)

In [3]:
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["y"], random_state=SEED)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df["y"], random_state=SEED)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

baseline = (test_df["y"] == 0).mean()
print(f"Majority-class baseline on test set: {baseline:.2%}")

train=4011  val=446  test=1115
Majority-class baseline on test set: 86.64%


## 4. Tokenize and build vocabulary (from training data only)

In [4]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " <url> ", text)
    text = re.sub(r"\d+", " <num> ", text)
    return re.findall(r"<url>|<num>|[a-z']+|[!?£$&]", text)

counter = Counter(tok for t in train_df["text"] for tok in tokenize(t))
PAD, UNK = 0, 1
itos = ["<pad>", "<unk>"] + [w for w, c in counter.items() if c >= 2]
stoi = {w: i for i, w in enumerate(itos)}
print("Vocab size:", len(itos))

MAX_LEN = 60
def encode(text):
    ids = [stoi.get(tok, UNK) for tok in tokenize(text)][:MAX_LEN]
    return ids if ids else [UNK]

print(tokenize(df["text"].iloc[2]))

Vocab size: 3224
['free', 'entry', 'in', '<num>', 'a', 'wkly', 'comp', 'to', 'win', 'fa', 'cup', 'final', 'tkts', '<num>', 'st', 'may', '<num>', 'text', 'fa', 'to', '<num>', 'to', 'receive', 'entry', 'question', 'std', 'txt', 'rate', 't', '&', "c's", 'apply', '<num>', 'over', '<num>', "'s"]


## 5. Dataset and DataLoaders

In [5]:
class SMSDataset(Dataset):
    def __init__(self, frame):
        self.x = [torch.tensor(encode(t)) for t in frame["text"]]
        self.y = torch.tensor(frame["y"].values, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

def collate(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    return pad_sequence(seqs, batch_first=True, padding_value=PAD), lengths, torch.stack(labels)

BATCH = 64
train_loader = DataLoader(SMSDataset(train_df), batch_size=BATCH, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(SMSDataset(val_df),   batch_size=BATCH, collate_fn=collate)
test_loader  = DataLoader(SMSDataset(test_df),  batch_size=BATCH, collate_fn=collate)

## 6. Model: Embedding → LSTM → Linear

Sequences are packed so the LSTM ignores padding; the final hidden state is fed to a linear classifier.

In [6]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=64, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        return self.fc(self.dropout(h_n[-1]))

model = LSTMClassifier(len(itos)).to(device)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

LSTMClassifier(
  (embedding): Embedding(3224, 64, padding_idx=0)
  (lstm): LSTM(64, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)
Parameters: 239746


## 7. Training

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

def evaluate(loader):
    model.eval()
    correct, total, preds, golds = 0, 0, [], []
    with torch.no_grad():
        for x, lengths, y in loader:
            x, y = x.to(device), y.to(device)
            p = model(x, lengths).argmax(1)
            correct += (p == y).sum().item()
            total += len(y)
            preds += p.tolist(); golds += y.tolist()
    return correct / total, preds, golds

EPOCHS = 10
best_val, best_state = 0.0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for x, lengths, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x, lengths), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(y)
    train_acc, _, _ = evaluate(train_loader)
    val_acc, _, _ = evaluate(val_loader)
    print(f"Epoch {epoch:2d} | loss {total_loss/len(train_loader.dataset):.4f} | train acc {train_acc:.2%} | val acc {val_acc:.2%}")
    if val_acc > best_val:
        best_val = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print(f"Best validation accuracy: {best_val:.2%}")

Epoch  1 | loss 0.2955 | train acc 96.09% | val acc 95.74%


Epoch  2 | loss 0.0898 | train acc 98.78% | val acc 97.31%


Epoch  3 | loss 0.0500 | train acc 99.10% | val acc 97.31%


Epoch  4 | loss 0.0407 | train acc 99.45% | val acc 97.98%


Epoch  5 | loss 0.0256 | train acc 99.53% | val acc 97.76%


Epoch  6 | loss 0.0235 | train acc 99.60% | val acc 97.09%


Epoch  7 | loss 0.0109 | train acc 99.83% | val acc 97.53%


Epoch  8 | loss 0.0083 | train acc 99.90% | val acc 98.21%


Epoch  9 | loss 0.0041 | train acc 99.93% | val acc 97.98%


Epoch 10 | loss 0.0042 | train acc 99.93% | val acc 97.98%
Best validation accuracy: 98.21%


## 8. Test accuracy

In [8]:
test_acc, preds, golds = evaluate(test_loader)
print(f"Baseline (majority class): {baseline:.2%}")
print(f"Target:                    96.00%")
print(f"LSTM test accuracy:        {test_acc:.2%}")
print("Beats target!" if test_acc >= 0.96 else ("Beats baseline" if test_acc > baseline else "Did not beat baseline"))
print()
print(classification_report(golds, preds, target_names=["ham", "spam"], digits=4))
print("Confusion matrix:\n", confusion_matrix(golds, preds))

Baseline (majority class): 86.64%
Target:                    96.00%
LSTM test accuracy:        98.48%
Beats target!

              precision    recall  f1-score   support

         ham     0.9887    0.9938    0.9912       966
        spam     0.9583    0.9262    0.9420       149

    accuracy                         0.9848      1115
   macro avg     0.9735    0.9600    0.9666      1115
weighted avg     0.9846    0.9848    0.9846      1115

Confusion matrix:
 [[960   6]
 [ 11 138]]


## 9. Try it on new messages

In [9]:
def predict(text):
    model.eval()
    ids = torch.tensor([encode(text)]).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(ids, torch.tensor([ids.shape[1]])), dim=1)[0]
    return ("spam" if probs[1] > 0.5 else "ham"), probs[1].item()

for msg in ["Congratulations! You've won a FREE £1000 gift card. Call 09061701461 now to claim!",
            "Hey, are we still meeting for lunch tomorrow?",
            "URGENT: your account has been selected for a cash prize. Text WIN to 80086"]:
    label, p = predict(msg)
    print(f"{label:4s} (p_spam={p:.3f})  {msg}")

spam (p_spam=1.000)  Congratulations! You've won a FREE £1000 gift card. Call 09061701461 now to claim!


ham  (p_spam=0.000)  Hey, are we still meeting for lunch tomorrow?
spam (p_spam=1.000)  URGENT: your account has been selected for a cash prize. Text WIN to 80086


## 10. Summary

- **Model:** Embedding (64) → single-layer `nn.LSTM` (64 hidden) → `nn.Linear` (2 classes), trained from scratch with Adam and cross-entropy.
- **Data:** stratified 72% / 8% / 20% train / validation / test split; vocabulary built from the training split only. The best epoch was picked on validation accuracy.
- **Result:** the test accuracy printed in Section 8 is compared to the 86.6% majority baseline and the 96% target.